<a href="https://colab.research.google.com/github/Jinal55/Hybrid-Deep-Learning-Framework-for-Climate-Change-Impact-Assessment-on-Agricultural-Crop-/blob/main/CropNet_Hybrid_CNN_BiLSTM_XGBoost_sem_3_rp_project_FIXED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Climate-Change-Aware Crop Yield Prediction on CropNet
### A Lightweight, Interpretable CNN–BiLSTM–XGBoost Hybrid Pipeline


In [1]:
# ============================================================
# Install dependencies (Colab)
# ============================================================
!pip install -q cropnet ecmwflibs xgboost shap opencv-python-headless seaborn urllib3 --upgrade

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.4/93.4 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.7/123.7 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.1/49.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.6/91.6 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 104.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.0/36.0 MB 28.7 MB/s eta 0:00:00


In [2]:
# Install urllib3 explicitly, as it seems to be a missing dependency for cropnet's download functionality.
!pip install -q urllib3

In [3]:
import os, json, math, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, roc_auc_score,
    mean_absolute_error, mean_squared_error, r2_score
)

import xgboost as xgb
import shap

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cuda


In [4]:
# ============================================================
# Global configuration — edit these for your own experiment
# ============================================================
class CFG:
    # ---- CropNet retrieval ----
    BASE_DIR   = "./content/CropNet-main.zip"          # Default, will be overridden
    CROP_TYPE  = "Soybean"
    IMAGE_TYPE = "AG"
    FIPS_CODES = ["19153", "17019", "29021", "20161", "31109",
                  "27143", "39159", "18157", "46099", "05007"]
    YEARS      = ["2018", "2019", "2020", "2021", "2022"]

    # ---- Sequence lengths after temporal alignment ----
    T_IMG   = 6
    T_CLIM  = 12
    N_CLIM_FEATS = 9

    IMG_SIZE = 224

    # ---- Splitting ----
    SPLIT_MODE = "spatiotemporal"
    TEST_YEARS = ["2022"]
    TEST_COUNTY_FRAC = 0.25

    # ---- Classification (yield-risk tiers) ----
    N_CLASSES = 3
    CLASS_NAMES = ["Low", "Medium", "High"]

    # ---- Training ----
    BATCH_SIZE = 8
    EPOCHS = 30
    LR = 1e-3
    WEIGHT_DECAY = 1e-4
    PATIENCE = 6
    CLS_LOSS_WEIGHT = 1.0
    REG_LOSS_WEIGHT = 0.5

    # ---- Demo / fallback ----
    # Set to True for synthetic data, False for real data.
    USE_SYNTHETIC = True
    # FIX: was 1 (a no-op given the old build_synthetic_dataset, which ignored this field
    # anyway). Now that build_synthetic_dataset() actually uses it (see §5), bumping this
    # to 6 turns 50 base (county,year) combinations into 300 samples -- enough that even
    # after the geography/time split, the test set has dozens of samples instead of 2.
    N_SYNTHETIC_SAMPLES_PER_COUNTY_YEAR = 6

cfg = CFG()

# Configure for real data as previously intended.
# This overrides the defaults defined in the CFG class for USE_SYNTHETIC and BASE_DIR.
cfg.USE_SYNTHETIC = False
cfg.BASE_DIR = "./cropnet_data"

# Ensure the base directory exists for the *new* BASE_DIR.
os.makedirs(cfg.BASE_DIR, exist_ok=True)

print(f"USE_SYNTHETIC is now set to: {cfg.USE_SYNTHETIC}")
print(f"BASE_DIR is now set to: {cfg.BASE_DIR}")

USE_SYNTHETIC is now set to: False
BASE_DIR is now set to: ./cropnet_data


The previous evaluation results were based on `USE_SYNTHETIC = True` as defined in the `CFG` class, meaning the models were trained and evaluated on artificially generated data. While useful for debugging the pipeline, synthetic data often lacks the complexity and realism required for robust model performance.

To address the 'improper' confusion matrix, CNN-BiLSTM graph performance, and XGBoost baseline results, we need to use real data. The following change will set `USE_SYNTHETIC` to `False` and adjust the `BASE_DIR` to a dedicated directory for downloaded `cropnet` data. Ensure your `BQ_PROJECT` ID (defined in an earlier cell) is correctly set if `cropnet` requires it for data retrieval.

In [5]:
# This cell is now redundant as its content has been merged into cell G1b4Av8Yrc2X.
# It is kept here as an empty cell to avoid changing cell IDs for subsequent cells.


## 4. CropNet Data Retrieval



In [6]:
def try_real_cropnet_download(cfg):
    '''Attempt to download & retrieve real CropNet data. Returns True on success.'''
    try:
        from cropnet.data_downloader import DataDownloader
        from cropnet.data_retriever import DataRetriever

        downloader = DataDownloader(target_dir=cfg.BASE_DIR)
        downloader.download_USDA(cfg.CROP_TYPE, fips_codes=cfg.FIPS_CODES, years=cfg.YEARS)
        downloader.download_Sentinel2(fips_codes=cfg.FIPS_CODES, years=cfg.YEARS,
                                       image_type=cfg.IMAGE_TYPE)
        downloader.download_HRRR(fips_codes=cfg.FIPS_CODES, years=cfg.YEARS)

        retriever = DataRetriever(base_dir=cfg.BASE_DIR)
        usda_data     = retriever.retrieve_USDA(crop_type=cfg.CROP_TYPE,
                                                  fips_codes=cfg.FIPS_CODES, years=cfg.YEARS)
        sentinel2_data = retriever.retrieve_Sentinel2(fips_codes=cfg.FIPS_CODES, years=cfg.YEARS,
                                                        image_type=cfg.IMAGE_TYPE)
        hrrr_data      = retriever.retrieve_HRRR(fips_codes=cfg.FIPS_CODES, years=cfg.YEARS)
        return usda_data, sentinel2_data, hrrr_data
    except Exception as e:
        print(f"[CropNet retrieval] Falling back to synthetic data. Reason: {e}")
        return None

real_data = None if cfg.USE_SYNTHETIC else try_real_cropnet_download(cfg)
if real_data is None:
    cfg.USE_SYNTHETIC = True
print("USE_SYNTHETIC =", cfg.USE_SYNTHETIC)

 ╭─▌▌Herbie─────────────────────────────────────────────╮
 │ INFO: Created a default config file.                 │
 │ You may view/edit Herbie's configuration here:       │
 │          /root/.config/herbie/config.toml            │
 ╰──────────────────────────────────────────────────────╯

[CropNet retrieval] Falling back to synthetic data. Reason: No module named 'pygrib'
USE_SYNTHETIC = True


## 5. Synthetic Data Generator

In [ ]:
def make_synthetic_image_sequence(T, size, greenness):
    '''Fake Sentinel-2 AG/NDVI-style RGB sequence. `greenness` in [0,1] biases toward green
    (healthier crop -> higher yield).'''
    imgs = np.zeros((T, size, size, 3), dtype=np.float32)
    for t in range(T):
        base = np.random.rand(size, size, 3).astype(np.float32) * 0.3
        # green channel emphasised by 'greenness', with some spatial smoothness
        g = greenness * (0.4 + 0.4 * t / max(T - 1, 1))
        field = np.random.rand(size, size).astype(np.float32)
        field = (field + np.roll(field, 5, axis=0) + np.roll(field, 5, axis=1)) / 3.0
        base[..., 1] += g * field
        base[..., 0] += 0.15 * field * (1 - greenness)
        imgs[t] = np.clip(base, 0, 1)
    return imgs  # (T, H, W, 3) in [0,1]

def make_synthetic_climate_sequence(T, months_favorable):
    '''9 WRF-HRRR-style params: [temp_avg, temp_max, temp_min, precip, humidity, wind,
    solar_rad, soil_temp, soil_moisture]. `months_favorable` shifts the growing-season months
    toward a yield-friendly regime.'''
    seq = np.zeros((T, 9), dtype=np.float32)
    for m in range(T):
        season = 1.0 - abs(m - 6) / 6.0          # peaks mid-year (growing season)
        favor = months_favorable if 3 <= m <= 8 else 0.0
        temp_avg = 15 + 15 * season + np.random.randn() * 2
        precip   = 60 + 40 * season * (0.5 + favor) + np.random.randn() * 10
        seq[m] = [
            temp_avg,
            temp_avg + 6 + np.random.randn(),
            temp_avg - 6 + np.random.randn(),
            max(precip, 0),
            55 + 15 * season + np.random.randn() * 5,
            3 + np.random.rand() * 4,
            150 + 150 * season + np.random.randn() * 20,
            temp_avg - 2 + np.random.randn(),
            0.2 + 0.15 * season * (0.5 + favor) + np.random.randn() * 0.02,
        ]
    return seq

def build_synthetic_dataset(cfg):
    '''
    FIX: previously this generated exactly ONE row per (fips, year), ignoring
    `cfg.N_SYNTHETIC_SAMPLES_PER_COUNTY_YEAR` entirely -- capping the dataset at
    len(FIPS_CODES) * len(YEARS) = 50 samples no matter what. We now actually draw
    `N_SYNTHETIC_SAMPLES_PER_COUNTY_YEAR` noisy replicates per (fips, year), which is
    what that config field was clearly meant to control. This directly enlarges every
    downstream split (train/val/test) so the test set isn't starved down to a handful
    of samples.
    '''
    rows = []
    rng = np.random.RandomState(SEED)
    n_reps = max(1, int(getattr(cfg, "N_SYNTHETIC_SAMPLES_PER_COUNTY_YEAR", 1)))
    for fips in cfg.FIPS_CODES:
        # each county has a latent baseline fertility (spatial signal to be learned)
        county_fertility = rng.uniform(0.3, 0.9)
        for year in cfg.YEARS:
            # global, shared climate-change drift by year -> temporal signal
            year_drift = (int(year) - 2017) * 0.02
            for rep in range(n_reps):
                favor = np.clip(county_fertility + year_drift + rng.normal(0, 0.1), 0, 1)
                imgs = make_synthetic_image_sequence(cfg.T_IMG, cfg.IMG_SIZE, greenness=favor)
                clim = make_synthetic_climate_sequence(cfg.T_CLIM, months_favorable=favor)

                # yield: nonlinear function of greenness + precip/temp balance + noise
                precip_score = clim[3:9, 3].mean() / 100.0
                temp_score   = 1 - abs(clim[3:9, 0].mean() - 24) / 24.0
                yield_val = 55 * favor + 20 * precip_score + 15 * temp_score
                yield_val += rng.normal(0, 3)
                yield_val = max(yield_val, 2.0)

                rows.append({
                    "fips": fips, "year": year,
                    "images": imgs.astype(np.float32),      # (T_IMG, H, W, 3)
                    "climate": clim.astype(np.float32),      # (T_CLIM, 9)
                    "yield": float(yield_val),
                })
    return rows

synthetic_rows = build_synthetic_dataset(cfg) if cfg.USE_SYNTHETIC else None
if synthetic_rows is not None:
    print(f"Built {len(synthetic_rows)} synthetic (county, year, rep) samples "
          f"across {len(cfg.FIPS_CODES)} counties x {len(cfg.YEARS)} years x "
          f"{cfg.N_SYNTHETIC_SAMPLES_PER_COUNTY_YEAR} replicate(s).")

## 6. Sentinel-2 Tensor Preprocessing



In [ ]:

# Converts raw Sentinel-2 AG/NDVI frames (whether from the real `cropnet` retriever or the
# synthetic generator) into normalized `float32` tensors of shape `(T_IMG, 3, 224, 224)`:
# resize, scale to `[0,1]`, then per-channel standardization with ImageNet-style statistics
# (a reasonable default for RGB satellite composites).
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def preprocess_image_sequence(img_seq_hwc, size=224):
    '''img_seq_hwc: (T, H, W, 3) float32 in [0,1] -> (T, 3, size, size) normalized tensor.'''
    T = img_seq_hwc.shape[0]
    out = np.zeros((T, 3, size, size), dtype=np.float32)
    for t in range(T):
        frame = img_seq_hwc[t]
        if frame.shape[0] != size or frame.shape[1] != size:
            # simple nearest-neighbour resize without extra deps
            ys = (np.linspace(0, frame.shape[0] - 1, size)).astype(int)
            xs = (np.linspace(0, frame.shape[1] - 1, size)).astype(int)
            frame = frame[ys][:, xs]
        frame = (frame - IMAGENET_MEAN) / IMAGENET_STD
        out[t] = frame.transpose(2, 0, 1)  # HWC -> CHW
    return out

## 7. Climate Sequence Construction


In [ ]:

# Builds the `(T_CLIM, 9)` WRF-HRRR monthly sequence per (county, year) and standardizes each of
# the 9 meteorological channels using statistics fit **only on the training split** (fit later,
# after the split, to avoid leakage — see §9).

def build_climate_sequence(raw_clim):
    '''raw_clim already (T_CLIM, 9) from retrieval/synthetic step; identity placeholder for
    real-data cleaning (drop NaNs, clip outliers).'''
    clim = np.nan_to_num(raw_clim, nan=0.0, posinf=0.0, neginf=0.0)
    clim = np.clip(clim, -3 * np.abs(clim).mean() - 1e-3, 1e6)
    return clim.astype(np.float32)

## 8. Temporal Alignment



In [ ]:

# Aligns the Sentinel-2 revisit sequence (14-day cadence, `T_IMG` frames) with the monthly
# climate sequence (`T_CLIM=12` months) so both describe the **same (county, year) growing
# season window**, and assembles the final per-sample record consumed by the `Dataset`.


def assemble_sample(row):
    imgs = preprocess_image_sequence(row["images"], size=cfg.IMG_SIZE)      # (T_IMG,3,H,W)
    clim = build_climate_sequence(row["climate"])                           # (T_CLIM,9)
    return {
        "fips": row["fips"],
        "year": int(row["year"]),
        "images": imgs,
        "climate": clim,
        "yield": row["yield"],
    }

if cfg.USE_SYNTHETIC:
    samples = [assemble_sample(r) for r in synthetic_rows]
else:
    # TODO: adapt to the objects returned by retriever.retrieve_* for real data —
    # iterate matching (fips, year) tuples across usda_data / sentinel2_data / hrrr_data
    # and call assemble_sample on each aligned triple.
    raise NotImplementedError("Wire up real-data alignment here once download succeeds.")

df_meta = pd.DataFrame([{"fips": s["fips"], "year": s["year"], "yield": s["yield"]}
                         for s in samples])
df_meta.head()

## 9. Yield-Risk Labels & Train/Val/Test Split (Geography + Time)

In [ ]:
def geography_time_split(samples, cfg):
    '''
    FIX #1 (already applied): the original "spatiotemporal" branch required a sample to be
    BOTH an unseen county AND the held-out year to count as "test" -- an intersection of two
    already-small subsets, which produced a 2-sample test set and an all-zero-looking
    confusion matrix. Fixed to an ADDITIVE definition (held-out counties -> test regardless
    of year; held-out year on remaining counties -> val).

    FIX #2 (this pass): even with a bigger test set, purely RANDOM county selection can by
    chance hold out counties that are all high-fertility (or all low-fertility), since in this
    synthetic generator each county has its own fixed latent `county_fertility` that drives
    yield far more than year-to-year climate noise does. That is exactly what happened with
    SEED=42: the 2 randomly-chosen held-out counties turned out to both be high-yield, so the
    test set collapsed to 28/30 samples in the "High" class -- a confusion matrix that's
    technically non-zero but still practically useless (2 of 3 classes barely represented).

    We now STRATIFY the held-out-county selection by each county's mean yield: rank counties
    by mean yield and take an evenly-spaced sample across that ranking (low, medium, and high
    counties all represented among the held-out set), instead of a uniform random draw.
    '''
    fips_all = sorted(set(s["fips"] for s in samples))
    n_test_counties = max(1, int(len(fips_all) * cfg.TEST_COUNTY_FRAC))

    # Rank counties by mean yield and take evenly spaced picks across the ranking so the
    # held-out set spans low/medium/high yield counties instead of clustering by chance.
    county_mean_yield = (
        pd.DataFrame([{"fips": s["fips"], "yield": s["yield"]} for s in samples])
        .groupby("fips")["yield"].mean().sort_values()
    )
    ranked_fips = county_mean_yield.index.tolist()
    stratified_idx = np.linspace(0, len(ranked_fips) - 1, n_test_counties).round().astype(int)
    held_out_counties = set(ranked_fips[i] for i in stratified_idx)

    test_years = set(int(y) for y in cfg.TEST_YEARS)

    def split_key(s):
        is_test_year = s["year"] in test_years
        is_held_county = s["fips"] in held_out_counties
        if cfg.SPLIT_MODE == "temporal":
            return "test" if is_test_year else "train"
        if cfg.SPLIT_MODE == "spatial":
            return "test" if is_held_county else "train"
        # spatiotemporal (ADDITIVE): held-out counties -> test (regardless of year);
        # held-out year on remaining (train) counties -> val; everything else -> train.
        if is_held_county:
            return "test"
        if is_test_year:
            return "val"
        return "train"

    tags = [split_key(s) for s in samples]
    # guarantee a non-trivial val set even for "temporal"/"spatial" modes: carve 15% of
    # train off (grouped by fips) for validation.
    if "val" not in tags:
        train_idx = [i for i, t in enumerate(tags) if t == "train"]
        groups = [samples[i]["fips"] for i in train_idx]
        gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
        tr_sub, val_sub = next(gss.split(train_idx, groups=groups))
        val_ids = set(train_idx[i] for i in val_sub)
        tags = [("val" if i in val_ids else t) for i, t in enumerate(tags)]

    print(f"Held-out (stratified-by-yield) test counties: {sorted(held_out_counties)}")
    return tags

split_tags = geography_time_split(samples, cfg)
for s, t in zip(samples, split_tags):
    s["split"] = t

print(pd.Series(split_tags).value_counts())

In [ ]:
# ---- Yield-risk class labels (tertiles fit on TRAIN only) ----
train_yields = np.array([s["yield"] for s in samples if s["split"] == "train"])
q1, q2 = np.quantile(train_yields, [1/3, 2/3])
print(f"Yield tertile cut points (train-only): q1={q1:.2f}, q2={q2:.2f}")

def yield_to_class(y):
    if y <= q1: return 0   # Low
    if y <= q2: return 1   # Medium
    return 2                # High

for s in samples:
    s["yield_class"] = yield_to_class(s["yield"])

# ---- Standardize climate features using TRAIN statistics only ----
train_clim_stack = np.concatenate([s["climate"] for s in samples if s["split"] == "train"], axis=0)
clim_scaler = StandardScaler().fit(train_clim_stack)

def scale_climate(clim):
    T, F_ = clim.shape
    return clim_scaler.transform(clim.reshape(-1, F_)).reshape(T, F_).astype(np.float32)

for s in samples:
    s["climate_scaled"] = scale_climate(s["climate"])

train_samples = [s for s in samples if s["split"] == "train"]
val_samples   = [s for s in samples if s["split"] == "val"]
test_samples  = [s for s in samples if s["split"] == "test"]
print(f"train={len(train_samples)}  val={len(val_samples)}  test={len(test_samples)}")
print("Class balance (train):", pd.Series([s['yield_class'] for s in train_samples]).value_counts().to_dict())

## 10. PyTorch `Dataset` / `DataLoader`

In [ ]:
class CropNetDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        images  = torch.from_numpy(s["images"])              # (T_IMG, 3, H, W)
        climate = torch.from_numpy(s["climate_scaled"])       # (T_CLIM, 9)
        y_class = torch.tensor(s["yield_class"], dtype=torch.long)
        y_val   = torch.tensor(s["yield"], dtype=torch.float32)
        return images, climate, y_class, y_val

train_ds = CropNetDataset(train_samples)
val_ds   = CropNetDataset(val_samples)
test_ds  = CropNetDataset(test_samples)

train_loader = DataLoader(train_ds, batch_size=cfg.BATCH_SIZE, shuffle=True,  drop_last=False)
val_loader   = DataLoader(val_ds,   batch_size=cfg.BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=cfg.BATCH_SIZE, shuffle=False)
print("Batches -> train:", len(train_loader), " val:", len(val_loader), " test:", len(test_loader))

## 11. Model — CNN Feature Extraction → BiLSTM Sequence Processing → Feature Fusion

In [ ]:
#  **`CNNEncoder`** — a small 4-block CNN applied *independently to every Sentinel-2 frame* in
#   the sequence, producing a 128-d embedding per frame (the last conv layer is also the
#   Grad-CAM target layer, see §15).
# * **`ImageBiLSTM`** — a bidirectional LSTM over the sequence of per-frame CNN embeddings,
#   capturing how the field's visual appearance evolves through the growing season.
# * **`ClimateBiLSTM`** — a second bidirectional LSTM over the raw (standardized) 9-channel
#   monthly climate sequence.
# * **`FusionHead`** — concatenates the two pooled BiLSTM representations and predicts both a
#   yield-risk **class** (for the confusion matrix) and the continuous yield **value**
#   (multi-task learning), giving the regression signal extra gradient information that helps
#   the classifier.


class CNNEncoder(nn.Module):
    '''Per-frame spatial feature extractor. Also exposes the last conv feature map for
    Grad-CAM.'''
    def __init__(self, out_dim=128):
        super().__init__()
        def block(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                nn.MaxPool2d(2)
            )
        self.block1 = block(3, 32)
        self.block2 = block(32, 64)
        self.block3 = block(64, 128)
        self.block4 = nn.Sequential(   # kept separate (no pooling) -> Grad-CAM target layer
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True)
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(128, out_dim)
        self._last_feat_map = None

    def forward(self, x):
        x = self.block1(x); x = self.block2(x); x = self.block3(x)
        x = self.block4(x)
        self._last_feat_map = x               # (B, 128, h, w) -> used by Grad-CAM
        if x.requires_grad:
            x.register_hook(self._save_grad)
        pooled = self.gap(x).flatten(1)        # (B, 128)
        return self.fc(pooled)                 # (B, out_dim)

    def _save_grad(self, grad):
        self._last_grad = grad


class ImageBiLSTM(nn.Module):
    def __init__(self, cnn_out_dim=128, hidden=64, out_dim=256):
        super().__init__()
        self.cnn = CNNEncoder(out_dim=cnn_out_dim)
        self.lstm = nn.LSTM(cnn_out_dim, hidden, batch_first=True, bidirectional=True)
        self.proj = nn.Linear(hidden * 2, out_dim)

    def forward(self, images):
        # images: (B, T, 3, H, W)
        B, T, C, H, W = images.shape
        feats = self.cnn(images.reshape(B * T, C, H, W))          # (B*T, cnn_out_dim)
        feats = feats.reshape(B, T, -1)                            # (B, T, cnn_out_dim)
        out, (h_n, _) = self.lstm(feats)                           # out: (B,T,2*hidden)
        pooled = out.mean(dim=1)                                   # mean pool over time
        return self.proj(pooled)                                   # (B, out_dim)


class ClimateBiLSTM(nn.Module):
    def __init__(self, n_feats=9, hidden=64, out_dim=256):
        super().__init__()
        self.lstm = nn.LSTM(n_feats, hidden, batch_first=True, bidirectional=True)
        self.proj = nn.Linear(hidden * 2, out_dim)

    def forward(self, climate):
        # climate: (B, T_CLIM, 9)
        out, _ = self.lstm(climate)
        pooled = out.mean(dim=1)
        return self.proj(pooled)


class FusionHead(nn.Module):
    def __init__(self, img_dim=256, clim_dim=256, hidden=128, n_classes=3,
                 use_image=True, use_climate=True):
        super().__init__()
        self.use_image = use_image
        self.use_climate = use_climate
        in_dim = (img_dim if use_image else 0) + (clim_dim if use_climate else 0)
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(hidden, hidden // 2), nn.ReLU(inplace=True)
        )
        self.cls_head = nn.Linear(hidden // 2, n_classes)
        self.reg_head = nn.Linear(hidden // 2, 1)

    def forward(self, img_repr, clim_repr):
        parts = []
        if self.use_image:   parts.append(img_repr)
        if self.use_climate: parts.append(clim_repr)
        fused = torch.cat(parts, dim=1)
        h = self.mlp(fused)
        return self.cls_head(h), self.reg_head(h).squeeze(1)


class HybridCNNBiLSTM(nn.Module):
    '''Full model: CNN feature extraction -> per-branch BiLSTM -> feature fusion -> heads.
    `use_image`/`use_climate` let ablation experiments disable a branch.'''
    def __init__(self, cfg, use_image=True, use_climate=True):
        super().__init__()
        self.use_image, self.use_climate = use_image, use_climate
        self.image_branch   = ImageBiLSTM()   if use_image   else None
        self.climate_branch = ClimateBiLSTM() if use_climate else None
        self.head = FusionHead(n_classes=cfg.N_CLASSES, use_image=use_image, use_climate=use_climate)

    def forward(self, images, climate):
        img_repr  = self.image_branch(images)   if self.use_image   else None
        clim_repr = self.climate_branch(climate) if self.use_climate else None
        return self.head(img_repr, clim_repr)

## 12. Training Loop (multi-task: classification + regression)

In [ ]:
def train_model(model, train_loader, val_loader, cfg, tag="hybrid"):
    model = model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=3)
    ce = nn.CrossEntropyLoss()
    mse = nn.MSELoss()

    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    best_val, best_state, patience_ctr = float("inf"), None, 0

    for epoch in range(cfg.EPOCHS):
        model.train()
        running = 0.0
        for images, climate, y_class, y_val in train_loader:
            images, climate = images.to(DEVICE), climate.to(DEVICE)
            y_class, y_val = y_class.to(DEVICE), y_val.to(DEVICE)

            opt.zero_grad()
            logits, y_hat = model(images, climate)
            loss = cfg.CLS_LOSS_WEIGHT * ce(logits, y_class) + cfg.REG_LOSS_WEIGHT * mse(y_hat, y_val)
            loss.backward()
            opt.step()
            running += loss.item() * images.size(0)
        train_loss = running / max(len(train_loader.dataset), 1)

        model.eval()
        v_running, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, climate, y_class, y_val in val_loader:
                images, climate = images.to(DEVICE), climate.to(DEVICE)
                y_class, y_val = y_class.to(DEVICE), y_val.to(DEVICE)
                logits, y_hat = model(images, climate)
                loss = cfg.CLS_LOSS_WEIGHT * ce(logits, y_class) + cfg.REG_LOSS_WEIGHT * mse(y_hat, y_val)
                v_running += loss.item() * images.size(0)
                correct += (logits.argmax(1) == y_class).sum().item()
                total += images.size(0)
        val_loss = v_running / max(total, 1)
        val_acc = correct / max(total, 1)
        sched.step(val_loss)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        print(f"[{tag}] epoch {epoch+1:02d}/{cfg.EPOCHS}  train_loss={train_loss:.4f}  "
              f"val_loss={val_loss:.4f}  val_acc={val_acc:.3f}")

        if val_loss < best_val - 1e-4:
            best_val, best_state, patience_ctr = val_loss, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            patience_ctr += 1
            if patience_ctr >= cfg.PATIENCE:
                print(f"[{tag}] early stopping at epoch {epoch+1}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history

hybrid_model = HybridCNNBiLSTM(cfg, use_image=True, use_climate=True)
hybrid_model, hybrid_history = train_model(hybrid_model, train_loader, val_loader, cfg, tag="hybrid")

## 13. XGBoost Baseline


In [ ]:

# Builds hand-crafted, **pooled** tabular features per sample (no learned CNN/LSTM):
# per-channel mean/std of climate over the season, and simple RGB summary statistics of the
# imagery sequence (mean/std per channel, a greenness proxy). An `XGBClassifier` predicts the
# yield-risk class (comparable confusion matrix / F1 to the hybrid model) and an
# `XGBRegressor` predicts the continuous yield value.



def make_tabular_features(s):
    clim = s["climate"]                       # (T_CLIM, 9) raw units
    clim_mean = clim.mean(axis=0)
    clim_std  = clim.std(axis=0)

    imgs = s["images"]                         # (T_IMG, C, H, W) from preprocess_image_sequence
    # Calculate mean and std for each of the 3 channels, averaging over time, height, and width
    img_mean = imgs.mean(axis=(0, 2, 3))        # (3,)
    img_std  = imgs.std(axis=(0, 2, 3))         # (3,)
    # Greenness: mean of (Green - Red) across time, height, and width
    greenness = (imgs[:, 1, :, :] - imgs[:, 0, :, :]).mean()
    feats = np.concatenate([clim_mean, clim_std, img_mean, img_std, [greenness]])
    return feats.astype(np.float32)

feat_names = ([f"clim_mean_{i}" for i in range(9)] + [f"clim_std_{i}" for i in range(9)] +
              ["img_mean_R", "img_mean_G", "img_mean_B", "img_std_R", "img_std_G", "img_std_B",
               "greenness"])

def build_tabular_split(samples):
    X = np.stack([make_tabular_features(s) for s in samples])
    y_cls = np.array([s["yield_class"] for s in samples])
    y_reg = np.array([s["yield"] for s in samples], dtype=np.float32)
    return X, y_cls, y_reg

X_train, ycls_train, yreg_train = build_tabular_split(train_samples)
X_val,   ycls_val,   yreg_val   = build_tabular_split(val_samples)
X_test,  ycls_test,  yreg_test  = build_tabular_split(test_samples)

xgb_clf = xgb.XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8,
    colsample_bytree=0.8, objective="multi:softprob", num_class=cfg.N_CLASSES,
    eval_metric="mlogloss", random_state=SEED
)
xgb_clf.fit(X_train, ycls_train, eval_set=[(X_val, ycls_val)], verbose=False)

xgb_reg = xgb.XGBRegressor(
    n_estimators=400, max_depth=4, learning_rate=0.05, subsample=0.8,
    colsample_bytree=0.8, objective="reg:squarederror", random_state=SEED
)
xgb_reg.fit(X_train, yreg_train, eval_set=[(X_val, yreg_val)], verbose=False)

print("XGBoost baseline trained.")

## 14. Evaluation — Confusion Matrix & Metrics (Hybrid vs. XGBoost)

In [ ]:
@torch.no_grad()
def predict_hybrid(model, loader):
    model.eval()
    all_logits, all_yhat, all_ycls, all_yval = [], [], [], []
    for images, climate, y_class, y_val in loader:
        images, climate = images.to(DEVICE), climate.to(DEVICE)
        logits, y_hat = model(images, climate)
        all_logits.append(logits.cpu().numpy())
        all_yhat.append(y_hat.cpu().numpy())
        all_ycls.append(y_class.numpy())
        all_yval.append(y_val.numpy())
    return (np.concatenate(all_logits), np.concatenate(all_yhat),
            np.concatenate(all_ycls), np.concatenate(all_yval))

hybrid_logits, hybrid_yhat, y_true_cls, y_true_val = predict_hybrid(hybrid_model, test_loader)
hybrid_pred_cls = hybrid_logits.argmax(1)
hybrid_probs = torch.softmax(torch.from_numpy(hybrid_logits), dim=1).numpy()

xgb_pred_cls = xgb_clf.predict(X_test)
xgb_probs = xgb_clf.predict_proba(X_test)
xgb_pred_val = xgb_reg.predict(X_test)

def classification_report_dict(y_true, y_pred, y_probs, tag):
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average="macro")
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="macro", zero_division=0)
    try:
        auc = roc_auc_score(y_true, y_probs, multi_class="ovr", average="macro")
    except Exception:
        auc = float("nan")
    print(f"--- {tag} (classification) ---")
    print(f"Accuracy={acc:.3f}  MacroF1={f1:.3f}  MacroPrec={prec:.3f}  MacroRec={rec:.3f}  MacroAUC={auc:.3f}")
    print(classification_report(y_true, y_pred, labels=list(range(cfg.N_CLASSES)), target_names=cfg.CLASS_NAMES, zero_division=0))
    return {"model": tag, "accuracy": acc, "macro_f1": f1, "macro_precision": prec,
            "macro_recall": rec, "macro_auc": auc}

def regression_report_dict(y_true, y_pred, tag):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"--- {tag} (regression) ---  MAE={mae:.3f}  RMSE={rmse:.3f}  R2={r2:.3f}")
    return {"model": tag, "MAE": mae, "RMSE": rmse, "R2": r2}

results_cls = [
    classification_report_dict(y_true_cls, hybrid_pred_cls, hybrid_probs, "Hybrid CNN-BiLSTM"),
    classification_report_dict(ycls_test,  xgb_pred_cls,    xgb_probs,    "XGBoost baseline"),
]
results_reg = [
    regression_report_dict(y_true_val, hybrid_yhat,   "Hybrid CNN-BiLSTM"),
    regression_report_dict(yreg_test,  xgb_pred_val,   "XGBoost baseline"),
]

df_results_cls = pd.DataFrame(results_cls)
df_results_reg = pd.DataFrame(results_reg)
df_results_cls

In [ ]:
# ---- Confusion matrices ----
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (y_t, y_p, title) in zip(
    axes,
    [(y_true_cls, hybrid_pred_cls, "Hybrid CNN-BiLSTM"), (ycls_test, xgb_pred_cls, "XGBoost baseline")]
):
    cm = confusion_matrix(y_t, y_p, labels=list(range(cfg.N_CLASSES)))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=cfg.CLASS_NAMES, yticklabels=cfg.CLASS_NAMES, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(f"Confusion Matrix — {title}")
plt.tight_layout(); plt.savefig("confusion_matrices.png", dpi=150); plt.show()

In [ ]:
# ---- Regression: actual vs predicted scatter ----
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (y_t, y_p, title) in zip(
    axes,
    [(y_true_val, hybrid_yhat, "Hybrid CNN-BiLSTM"), (yreg_test, xgb_pred_val, "XGBoost baseline")]
):
    ax.scatter(y_t, y_p, alpha=0.6, edgecolor="k")
    lims = [min(y_t.min(), y_p.min()), max(y_t.max(), y_p.max())]
    ax.plot(lims, lims, "r--", linewidth=1)
    ax.set_xlabel("Actual yield"); ax.set_ylabel("Predicted yield"); ax.set_title(title)
plt.tight_layout(); plt.savefig("regression_scatter.png", dpi=150); plt.show()

In [ ]:
# ---- Training curves ----
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hybrid_history["train_loss"], label="train loss")
axes[0].plot(hybrid_history["val_loss"], label="val loss")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].legend(); axes[0].set_title("Hybrid model loss")
axes[1].plot(hybrid_history["val_acc"], label="val accuracy", color="green")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("accuracy"); axes[1].legend(); axes[1].set_title("Hybrid model val accuracy")
plt.tight_layout(); plt.savefig("training_curves.png", dpi=150); plt.show()

## 15. Ablation Experiments

Compares four variants on the same test split:

1. **Image-only** — `HybridCNNBiLSTM(use_image=True, use_climate=False)`
2. **Climate-only** — `HybridCNNBiLSTM(use_image=False, use_climate=True)`
3. **Full fusion** — both branches (the model trained in §12)
4. **XGBoost baseline** — pooled hand-crafted features (§13)

In [ ]:
def run_ablation_variant(use_image, use_climate, tag):
    m = HybridCNNBiLSTM(cfg, use_image=use_image, use_climate=use_climate)
    m, hist = train_model(m, train_loader, val_loader, cfg, tag=tag)
    logits, yhat, y_t_cls, y_t_val = predict_hybrid(m, test_loader)
    pred_cls = logits.argmax(1)
    probs = torch.softmax(torch.from_numpy(logits), dim=1).numpy()
    row = classification_report_dict(y_t_cls, pred_cls, probs, tag)
    reg_row = regression_report_dict(y_t_val, yhat, tag)
    row.update({"RMSE": reg_row["RMSE"], "MAE": reg_row["MAE"], "R2": reg_row["R2"]})
    return row, m

ablation_rows = []
row, _ = run_ablation_variant(True, False, "Image-only (CNN+BiLSTM)")
ablation_rows.append(row)
row, _ = run_ablation_variant(False, True, "Climate-only (BiLSTM)")
ablation_rows.append(row)

full_row = dict(results_cls[0]); full_row.update({"RMSE": results_reg[0]["RMSE"],
                                                    "MAE": results_reg[0]["MAE"], "R2": results_reg[0]["R2"]})
full_row["model"] = "Full fusion (CNN+BiLSTM x2)"
ablation_rows.append(full_row)

xgb_row = dict(results_cls[1]); xgb_row.update({"RMSE": results_reg[1]["RMSE"],
                                                  "MAE": results_reg[1]["MAE"], "R2": results_reg[1]["R2"]})
xgb_row["model"] = "XGBoost baseline"
ablation_rows.append(xgb_row)

df_ablation = pd.DataFrame(ablation_rows)
df_ablation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.barplot(data=df_ablation, x="model", y="accuracy", ax=axes[0], palette="viridis")
axes[0].set_title("Ablation — Classification Accuracy"); axes[0].tick_params(axis='x', rotation=25)
sns.barplot(data=df_ablation, x="model", y="RMSE", ax=axes[1], palette="magma")
axes[1].set_title("Ablation — Regression RMSE (lower is better)"); axes[1].tick_params(axis='x', rotation=25)
plt.tight_layout(); plt.savefig("ablation_results.png", dpi=150); plt.show()

## 16. SHAP — Explaining the XGBoost Baseline

`shap.TreeExplainer` gives exact, fast Shapley values for the tree-based classifier, showing
which pooled climate/imagery features (e.g., seasonal precipitation mean, greenness proxy)
push a county-year toward Low/Medium/High yield-risk predictions.

In [ ]:
explainer = shap.TreeExplainer(xgb_clf)
shap_values = explainer.shap_values(X_test)   # list-of-arrays (per class) or (N, F, C) depending on version

X_test_df = pd.DataFrame(X_test, columns=feat_names)

# Handle both SHAP API shapes (older: list per class; newer: single (N,F,C) array)
if isinstance(shap_values, list):
    sv_for_plot = shap_values[cfg.N_CLASSES - 1]   # explain the "High yield" class
else:
    sv_for_plot = shap_values[..., cfg.N_CLASSES - 1]

plt.figure()
shap.summary_plot(sv_for_plot, X_test_df, show=False)
plt.title("SHAP Summary — features driving 'High yield' predictions (XGBoost)")
plt.tight_layout(); plt.savefig("shap_summary.png", dpi=150); plt.show()

plt.figure()
shap.summary_plot(sv_for_plot, X_test_df, plot_type="bar", show=False)
plt.title("Mean |SHAP value| by feature")
plt.tight_layout(); plt.savefig("shap_bar.png", dpi=150); plt.show()

## 17. Grad-CAM — Explaining the CNN Image Branch

A manual Grad-CAM implementation targeting `CNNEncoder.block4` (the last convolutional block
before global pooling). For a chosen test sample and its predicted class, we backpropagate
the class logit to the last feature map, weight each channel by its average gradient, and
overlay the resulting saliency map on the original Sentinel-2 frame to show **which parts of
the field** most influenced the yield-risk prediction.

In [ ]:
def grad_cam_for_frame(model, image_chw, climate_seq, target_class=None):
    '''image_chw: single normalized frame (3,H,W) tensor (already preprocessed).
    climate_seq: (T_CLIM, 9) tensor for the same sample (needed since the model is multi-modal).
    Returns (heatmap [H,W] in [0,1], predicted_class:int).'''

    # Temporarily set model to train mode for both forward and backward pass.
    # This is necessary for cuDNN RNNs to compute gradients, which otherwise raise a RuntimeError.
    # Note: Performing the forward pass in train mode means dropout layers are active
    # and BatchNorm uses batch statistics. This might slightly alter the exact logits
    # compared to a pure eval-mode inference, but it's a necessary compromise to enable Grad-CAM
    # with cuDNN RNNs.
    model.train()

    cnn = model.image_branch.cnn
    img_batch = image_chw.unsqueeze(0).unsqueeze(0).to(DEVICE).requires_grad_(True)  # (1,1,3,H,W)
    clim_batch = climate_seq.unsqueeze(0).to(DEVICE)

    logits, _ = model(img_batch, clim_batch) # Forward pass in train mode
    if target_class is None:
        target_class = int(logits.argmax(1).item())

    model.zero_grad()
    logits[0, target_class].backward() # Backward pass in train mode

    # Switch back to eval mode for consistency with normal inference after Grad-CAM computation
    model.eval()

    feat_map = cnn._last_feat_map            # (1, 128, h, w)  captured during forward
    grads    = cnn._last_grad                # (1, 128, h, w)  captured via hook
    weights  = grads.mean(dim=(2, 3), keepdim=True)          # global-average-pool the gradients
    cam = F.relu((weights * feat_map).sum(dim=1)).squeeze(0)  # (h, w)
    cam = cam.detach().cpu().numpy()
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    cam = np.array(
        [[cam[int(i * cam.shape[0] / cfg.IMG_SIZE), int(j * cam.shape[1] / cfg.IMG_SIZE)]
          for j in range(cfg.IMG_SIZE)] for i in range(cfg.IMG_SIZE)]
    )  # nearest-neighbour upsample to full resolution (dependency-free)
    return cam, target_class

def denormalize_for_display(img_chw):
    img = img_chw.numpy().transpose(1, 2, 0)
    img = img * IMAGENET_STD + IMAGENET_MEAN
    return np.clip(img, 0, 1)

n_examples = min(3, len(test_samples))
fig, axes = plt.subplots(n_examples, 2, figsize=(8, 4 * n_examples))
if n_examples == 1:
    axes = axes[None, :]

for i in range(n_examples):
    s = test_samples[i]
    frame = torch.from_numpy(s["images"][-1])                 # last frame in the season
    clim  = torch.from_numpy(s["climate_scaled"])
    cam, pred_class = grad_cam_for_frame(hybrid_model, frame, clim)
    disp = denormalize_for_display(frame)

    axes[i, 0].imshow(disp); axes[i, 0].set_title(f"Original \u2014 true={cfg.CLASS_NAMES[s['yield_class']]}")
    axes[i, 0].axis("off")
    axes[i, 1].imshow(disp); axes[i, 1].imshow(cam, cmap="jet", alpha=0.45)
    axes[i, 1].set_title(f"Grad-CAM \u2014 pred={cfg.CLASS_NAMES[pred_class]}")
    axes[i, 1].axis("off")

plt.tight_layout(); plt.savefig("gradcam_examples.png", dpi=150); plt.show()

## 18. Results Summary

In [ ]:
print("=== Classification results (test split) ===")
display(df_results_cls)
print("\n=== Regression results (test split) ===")
display(df_results_reg)
print("\n=== Ablation results (test split) ===")
display(df_ablation)

df_results_cls.to_csv("classification_results.csv", index=False)
df_results_reg.to_csv("regression_results.csv", index=False)
df_ablation.to_csv("ablation_results.csv", index=False)
print("\nSaved: classification_results.csv, regression_results.csv, ablation_results.csv")
print("Saved figures: confusion_matrices.png, regression_scatter.png, training_curves.png, "
      "ablation_results.png, shap_summary.png, shap_bar.png, gradcam_examples.png")

## 19. Notes for Moving from Synthetic → Real CropNet Data


In [ ]:
# NOTE: this cell was an exact duplicate of the SHAP analysis in cell 37 (§16),
# re-running the same TreeExplainer computation and overwriting the same PNGs a
# second time for no benefit. Left empty to avoid renumbering subsequent cells.
